# Lab 22-08 · Power Management

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate
import pmic as pm


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · DVS ramp tracking

*Predict* the output's lead over a 0.2 mV/µs reference ramp for Example 22-07.3's loop, then at 1 mV/µs.

In [2]:
R1, C1, C2 = 10e3, 11.236714e-9, 0.6740832e-9
w_i = 1 / (R1 * (C1 + C2))
for slew in (200.0, 1000.0):
    print(f"slew {slew/1e3:.1f} mV/us: predicted lead {pm.ramp_lead(slew, w_i, 1.0, 12.0)*1e3:.1f} mV")
s = ngspice("dvs.cir")
print({kk: s[kk] for kk in ("lead", "vfinal")})

slew 0.2 mV/us: predicted lead 21.8 mV
slew 1.0 mV/us: predicted lead 109.2 mV
{'lead': 0.02183635, 'vfinal': 1.2}


## Part 2 · Soft start

*Predict* the energy each switch dissipates, both peak currents, and the soft-started domain's time to 0.99 V.

In [3]:
t_ss, e_sw = pm.soft_start(100e-6, 1.0, 0.1)
print(f"soft start: {t_ss*1e3:.2f} ms to full charge, switch energy {e_sw*1e6:.1f} uJ either way; hard peak {1.0/0.1:.0f} A")
s = ngspice("softstart.cir")
print({kk: s[kk] for kk in ("ehard", "esoft", "ipk_hard", "ipk_soft", "tcharge")})

soft start: 1.00 ms to full charge, switch energy 50.0 uJ either way; hard peak 10 A


{'ehard': 4.99848e-05, 'esoft': 4.99999e-05, 'ipk_hard': 9.999, 'ipk_soft': 0.1, 'tcharge': 0.00099}


## Part 3 · Light-load modes

*Predict* which mode wins at 1 mA and at 300 mA for Example 22-08.2's buck.

In [4]:
i = np.logspace(-5, 0, 300)
po = 1.8 * i
e_pwm = pm.efficiency(po, pm.pwm_loss(i, 20e-3, 0.2))
e_pfm = pm.efficiency(po, pm.pfm_loss(i, 3.6, 1e-6, 0.12, 0.6))
ix = pm.pfm_pwm_crossover(20e-3, 0.2, 3.6, 1e-6, 0.12, 0.6)
plt.figure()
plt.semilogx(i * 1e3, 100 * e_pwm, label="forced PWM"); plt.semilogx(i * 1e3, 100 * e_pfm, label="PFM")
plt.axvline(ix * 1e3, color="0.5", ls=":"); plt.xlabel("load (mA)"); plt.ylabel("efficiency (%)"); plt.legend(); plt.show()
print(f"crossover {ix*1e3:.0f} mA")

crossover 119 mA


## Part 4 · The battery budget

*Predict* which state dominates the fitness band's day, and which of three halvings helps most.

In [5]:
band = [(50e-6, 82800, 0.80), (15e-3, 3300, 0.88), (150e-3, 300, 0.90)]
e_bat = pm.battery_energy(300, 3.85)
e, eq, tot = pm.daily_energy(band, 3.85, 1e-6)
print("per state (J):", [round(x, 2) for x in e], "PMIC", round(eq, 3), "total", round(tot, 1), f"-> {e_bat/tot:.1f} days")
cases = {
    "baseline": (band, 1e-6),
    "half sleep power": ([(25e-6, 82800, 0.80)] + band[1:], 1e-6),
    "half quiescent current": (band, 0.5e-6),
    "half active-state losses": ([band[0]] + [(p, t, 1 / (1 + (1 / eta - 1) / 2)) for p, t, eta in band[1:]], 1e-6),
}
for name, (st, iq) in cases.items():
    print(f"{name:26s} {e_bat / pm.daily_energy(st, 3.85, iq)[2]:.2f} days")

per state (J): [5.18, 56.25, 50.0] PMIC 0.333 total 111.8 -> 37.2 days
baseline                   37.21 days
half sleep power           38.09 days
half quiescent current     37.26 days
half active-state losses   39.27 days


## Part 5 · Gating when hot

*Predict* how Problem 22-08.11's break-even time changes from 25 °C to 85 °C.

In [6]:
temps = np.linspace(0, 100, 101)
tbe = [pm.gating_breakeven(50e-9, 0.75, 0.5e-6, pm.leakage_at(2e-3, t)) for t in temps]
plt.figure(); plt.semilogy(temps, np.array(tbe) * 1e6)
plt.xlabel("temperature (C)"); plt.ylabel("break-even time (us)"); plt.show()
for t in (25, 85):
    print(f"{t} C: {pm.gating_breakeven(50e-9, 0.75, 0.5e-6, pm.leakage_at(2e-3, t))*1e6:.1f} us")

25 C: 264.1 us
85 C: 50.0 us


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.